<a href="https://colab.research.google.com/github/arratebello/arratebello/blob/main/Proyecto_Algoritmos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Proyecto de programación - Algoritmos de optimización**<br>

Nombre y Apellidos: **Julián Samblás Caballero, Arrate Bello Martija**    <br>
Url: https://github.com/.../03MAIR---Algoritmos-de-Optimizacion---2019/tree/master/SEMINARIO<br>          

### Problema: **Organizar los horarios de partidos de La Liga**

**Descripción del problema:**

Desde la La Liga de fútbol profesional se pretende organizar los horarios de los partidos de liga de cada jornada. Se conocen algunos datos que nos deben llevar a diseñar un algoritmo que realice la asignación de los partidos a los horarios de forma que maximice la audiencia.

**1. Horarios Disponibles**
Los horarios disponibles se conocen a priori y son los siguientes:
*   **Viernes:** 20h
*   **Sábado:** 12h, 16h, 18h, 20h
*   **Domingo:** 12h, 16h, 18h, 20h
*   **Lunes:** 20h

**2. Categorías de los Equipos y Audiencia Base**
En primer lugar se clasifican los equipos en tres categorías según el número de seguidores (que tiene relación directa con la audiencia). Hay 3 equipos en la categoría A, 11 equipos de categoría B y 6 equipos de categoría C.

Se conoce estadísticamente la audiencia que genera cada partido según los equipos que se enfrentan y en horario de sábado a las 20h (el mejor en todos los casos):

| | Categoría A | Categoría B | Categoría C |
| :--- | :---: | :---: | :---: |
| **Categoría A** | 2 Millones | 1.3 Millones | 1 Millón |
| **Categoría B** | - | 0.9 Millones | 0.75 Millones |
| **Categoría C** | - | - | 0.47 Millones* |
*(Nota: La audiencia base para un partido C-C de 0.47 Millones se deduce de la tabla de cálculos de ejemplo de la jornada).*

**3. Ponderación por Horario y Restricciones Obligatorias**
Si el horario del partido no se realiza a las 20 horas del sábado, se sabe que se reduce la audiencia según los coeficientes de la siguiente tabla.
**Restricción obligatoria:** Debemos asignar obligatoriamente siempre un partido el viernes y un partido el lunes.

| Horario | Viernes | Sábado | Domingo | Lunes |
| :---: | :---: | :---: | :---: | :---: |
| **12h** | - | 0.55 | 0.45 | - |
| **16h** | - | 0.7 | 0.75 | - |
| **18h** | - | 0.8 | 0.85 | - |
| **20h** | 0.4 | 1 | 1 | 0.4 |

**4. Penalización por Coincidencias**
Es posible la coincidencia de horarios pero en este caso la audiencia de cada partido se verá afectada y se estima que se reduce en porcentaje según la siguiente tabla dependiendo del número de coincidencias:

| Coincidencias | Reducción (%) |
| :---: | :---: |
| **0** | 0% |
| **1** | 25% |
| **2** | 45% |
| **3** | 60% |
| **4** | 70% |
| **5** | 75% |
| **6** | 78% |
| **7** | 80% |
| **8** | 80% |

El cálculo final de audiencia por partido será: `(Base * Ponderación) * (1 - Porcentaje de reducción por coincidencia)`.




(*) La respuesta es obligatoria


Antes de comenzar, trasladamos a código los datos del enunciado y definimos las funciones base que utilizaremos a lo largo del notebook.

In [17]:
import random
import itertools
import math
from collections import Counter
import time

Declaramos las estructuras de datos con los horarios disponibles, la tabla de ponderaciones, la audiencia base según las categorías de los equipos y los porcentajes de reducción por coincidencia.

In [18]:
# Los 10 horarios disponibles en una jornada
DIAS_HORARIOS = [
    ('Viernes', 20),
    ('Sabado', 12), ('Sabado', 16), ('Sabado', 18), ('Sabado', 20),
    ('Domingo', 12), ('Domingo', 16), ('Domingo', 18), ('Domingo', 20),
    ('Lunes', 20),
]
IDX_VIERNES = 0   # posición de ('Viernes', 20) en DIAS_HORARIOS
IDX_LUNES = 9     # posición de ('Lunes', 20) en DIAS_HORARIOS

# Tabla de ponderación por horario
PONDERACION = {
    ('Viernes', 20): 0.40,
    ('Sabado', 12): 0.55, ('Sabado', 16): 0.70, ('Sabado', 18): 0.80, ('Sabado', 20): 1.00,
    ('Domingo', 12): 0.45, ('Domingo', 16): 0.75, ('Domingo', 18): 0.85, ('Domingo', 20): 1.00,
    ('Lunes', 20): 0.40,
}

# Tabla de audiencia base por categorías enfrentadas
# Usamos la tupla ordenada alfabéticamente como clave porque la tabla es simétrica.
BASE_AUDIENCIA = {
    ('A', 'A'): 2.00,
    ('A', 'B'): 1.30,
    ('A', 'C'): 1.00,
    ('B', 'B'): 0.90,
    ('B', 'C'): 0.75,
    ('C', 'C'): 0.47,
}

# Tabla de reducción de audiencia según el número de coincidencias
REDUCCION_COINCIDENCIA = {0: 0.00, 1: 0.25, 2: 0.45, 3: 0.60, 4: 0.70,
                          5: 0.75, 6: 0.78, 7: 0.80, 8: 0.80}


Definimos funciones de apoyo para calcular la audiencia, aplicar la penalización por solapamiento (`factor_coincidencia`), comprobar si una asignación cumple la restricción obligatoria del viernes y el lunes (`es_valida`) y calcular nuestra valor de la función objetivo (`audiencia_total`).

In [19]:
def audiencia_base(cat1, cat2):
    '''Audiencia base (en millones) de un partido según las categorías de los equipos.'''
    return BASE_AUDIENCIA[tuple(sorted([cat1, cat2]))]

def factor_coincidencia(n_coincidencias):
    '''Factor multiplicativo (entre 0 y 1) según el número de OTROS partidos
    que comparten el mismo horario.'''
    n_coincidencias = min(n_coincidencias, 8)
    return 1 - REDUCCION_COINCIDENCIA[n_coincidencias]

def audiencia_total(asignacion, partidos):
    '''Función objetivo: audiencia total (en millones) de la jornada.
    asignacion: lista de longitud n, asignacion[i] = índice de horario (0..9) del partido i.
    partidos:   lista de tuplas (categoria_1, categoria_2), una por partido.
    '''
    ocupacion = Counter(asignacion)
    total = 0.0
    for i, horario_idx in enumerate(asignacion):
        base = audiencia_base(*partidos[i])
        pond = PONDERACION[DIAS_HORARIOS[horario_idx]]
        n_coincidencias = ocupacion[horario_idx] - 1
        total += base * pond * factor_coincidencia(n_coincidencias)
    return total

def es_valida(asignacion):
    '''Comprueba la restricción obligatoria: al menos un partido en viernes y otro en lunes.'''
    return (IDX_VIERNES in asignacion) and (IDX_LUNES in asignacion)

<br>**1. (*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>**

**¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones?**




**Respuesta:**

El problema consiste en asignar a cada uno de los $n=10$ partidos uno de los $k=10$ horarios disponible. El enunciado permite explícitamente que varios partidos coincidan en el mismo horario, es decir, no hace falta que cada horario se use como máximo una vez. Por lo tanto, no es una asignación biyectiva y cada partido elige su horario de forma independiente de los demás.

Por una parte, si contamos las posibilidades de asignación que hay sin tener en cuenta las restricciones, esto es, sin exigir partido el viernes ni el lunes, para cada partido tenemos 10 posibilidades de horario. Como los partidos son distintos entre sí y un mismo horario puede usarse para más de un partido simultáneamente, se trata de tomar 10 decisiones independientes (una por partido) eligiendo entre 10 opciones cada vez, es decir, variaciones con repetición de 10 elementos tomados de 10 en 10
. Por lo tanto, el total de combinaciones posibles es:

$$VR_{10,10}=10^{10} = 10.000.000.000 \text{ posibilidades.}$$

<br>

Por otra parte, si tenemos en cuenta las restricciones, es decir, si exigimos que tiene que haber al menos un partido el viernes y al menos uno el lunes, usamos el Principio de inclusión-exclusión para calcular el número de posibilidades.
Sea $U$ el conjunto de todas las asignaciones posibles, $A$ el conjunto de asignaciones sin ningún partido el viernes y $B$ el conjunto de asignaciones sin ningún partido el lunes.
Teniendo en cuenta estos conjuntos, lo que buscamos es contar las asignaciones válidas, es decir, el total menos las opciones donde falta el viernes, falta el lunes, o faltan ambos. Aplicando el principio de inclusión-exclusión:
$$ |U| - |A \cup B| = |U| - (|A|+|B|-|A \cap B|)$$

En el anterior párrafo, hemos calculado que $|U|=10.000.000.000$. Además, $|A| = 9^{10}$ ya que cada partido elige entre los 9 horarios que no son viernes y análogamente para el lunes tenemos que $|B| = 9^{10}$. Por último, $|A \cap B| = 8^{10}$, ya que es el caso donde no hay partidos ni el viernes ni en lunes (nos quedan 8 horarios disponibles).

Por lo tanto, el total de posibiliades teniendo en cuenta las restricciones son:
$$|U| - |A \cup B| = 10^{10} - \big(9^{10}+9^{10}-8^{10}\big) = 10^{10} - 2\cdot 9^{10} + 8^{10}=4.100.173.022 \text{ posibilidades.}$$

Este espacio de soluciones es el que hace que la fuerza bruta sea inviable para el tamaño real del problema (lo vamos a comprobar en la pregunta 5).

<br>**Modelo para el espacio de soluciones<br>**
**2. (*) ¿Cual es la estructura de datos que mejor se adapta al problema? Arguméntalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguméntalo)**


Lo que necesitamos representar es una solución al problema, es decir, a qué horario va cada partido. Como tenemos 10 partidos y 10 horarios posibles, una solución no es más que decir "el partido 0 va al horario X, el partido 1 va al horario Y", etc.

La forma más sencilla de representar esto es con una **lista de 10 enteros**. Cada posición de la lista corresponde a un partido, y el valor que contiene es el índice del horario asignado (un número del 0 al 9 según la tabla `DIAS_HORARIOS`).

Por ejemplo, para los horarios:

| Índice | Día - Hora |
|:---:|:---|
| 0 | Viernes 20h |
| 1 | Sábado 12h |
| 2 | Sábado 16h |
| ... | ... |
| 9 | Lunes 20h |

Una asignación como `[4, 8, 3, 6, 7, 2, 5, 1, 0, 9]` significa que el partido 0 se juega en el horario 4 (Sábado 20h), el partido 1 en el horario 8 (Domingo 20h), y así sucesivamente.

He elegido esta estructura por varias razones:

- Es simple y directa: con 10 números representamos toda una jornada.
- Permite repeticiones: varios partidos pueden tener el mismo índice (mismo horario), que es exactamente lo que contempla el enunciado con las penalizaciones por coincidencia.
- Es compatible con las funciones que ya tenemos definidas: `audiencia_total()` recibe una lista de índices, y `es_valida()` simplemente comprueba si el 0 y el 9 aparecen en la lista.
- Funciona muy bien con `itertools.product(range(10), repeat=10)` para generar todas las combinaciones en fuerza bruta.

Otras opciones como un diccionario `{partido: horario}` serían equivalentes pero más pesadas, y una matriz binaria de 10×10 sería innecesariamente compleja para lo que necesitamos.

In [20]:
# Ejemplo de la estructura de datos
partidos_ejemplo = [
    ('A', 'A'),  # 0: Real Madrid – Barcelona
    ('A', 'B'),  # 1: Atlético – Sevilla
    ('A', 'B'),  # 2: Real Madrid – Valencia
    ('B', 'B'),  # 3: Betis – Celta
    ('B', 'B'),  # 4: Athletic – Real Sociedad
    ('B', 'C'),  # 5: Villarreal – Getafe
    ('B', 'C'),  # 6: Osasuna – Mallorca
    ('C', 'C'),  # 7: Espanyol – Alavés
    ('C', 'C'),  # 8: Valladolid – Cádiz
    ('C', 'C'),  # 9: Granada – Almería
]

# Nuestra solución es simplemente una lista de índices
asignacion = [4, 8, 3, 6, 7, 2, 5, 1, 0, 9]

# Veamos qué significa cada valor
print("Interpretación de la asignación:")
print("-" * 50)
for i, h in enumerate(asignacion):
    dia, hora = DIAS_HORARIOS[h]
    cat1, cat2 = partidos_ejemplo[i]
    print(f"  Partido {i} ({cat1} vs {cat2}) → {dia} {hora}h")

print(f"\n¿Es válida? {es_valida(asignacion)}")
print(f"Audiencia total: {audiencia_total(asignacion, partidos_ejemplo):.4f} millones")

Interpretación de la asignación:
--------------------------------------------------
  Partido 0 (A vs A) → Sabado 20h
  Partido 1 (A vs B) → Domingo 20h
  Partido 2 (A vs B) → Sabado 18h
  Partido 3 (B vs B) → Domingo 16h
  Partido 4 (B vs B) → Domingo 18h
  Partido 5 (B vs C) → Sabado 16h
  Partido 6 (B vs C) → Domingo 12h
  Partido 7 (C vs C) → Sabado 12h
  Partido 8 (C vs C) → Viernes 20h
  Partido 9 (C vs C) → Lunes 20h

¿Es válida? True
Audiencia total: 7.2770 millones


<br>**3. Según el modelo para el espacio de soluciones<br>**
**(*)¿Cual es la función objetivo?<br>**
**(*)¿Es un problema de maximización o minimización?**

**Respuesta:**

Es un problema de **maximización**: queremos que la audiencia total de la jornada sea lo más alta posible.

La **función objetivo** es la función `audiencia_total(asignacion, partidos)` que ya está definida en la celda de datos. Lo que hace es sumar la audiencia de cada partido, que se calcula multiplicando tres factores:

1. Audiencia base: depende de las categorías de los equipos que se enfrentan (un A vs A genera 2M, un C vs C solo 0.47M, etc.). Este valor es fijo para cada partido.

2. Ponderación del horario: un coeficiente entre 0 y 1 que penaliza los horarios malos. Sábado/Domingo a las 20h tienen ponderación 1.0 (sin penalización), mientras que Viernes y Lunes a las 20h solo tienen 0.4.

3. Factor de coincidencia: si hay varios partidos en el mismo horario, la audiencia de cada uno baja. Con 1 coincidencia se reduce un 25%, con 2 un 45%, etc. Este es el factor más interesante porque hace que la audiencia de un partido dependa de dónde están colocados los demás.

Es decir, la audiencia de cada partido es:

$$\text{Audiencia del partido } i = \text{Base}(i) \times \text{Ponderación}(horario_i) \times (1 - \text{Reducción}(coincidencias_i))$$

Y la función objetivo es la suma de todas:

$$f(\mathbf{x}) = \sum_{i=0}^{9} \text{Audiencia del partido } i$$

Por ejemplo, un partido A vs A en Sábado 20h sin coincidencias da $2.00 \times 1.00 \times 1.00 = 2.00$M. El mismo partido en Viernes 20h con 1 coincidencia da $2.00 \times 0.40 \times 0.75 = 0.60$M. La diferencia es enorme, y por eso la asignación importa tanto.

---



In [21]:
# Desglose de la función objetivo para ver cómo funciona

asignacion = [4, 8, 3, 6, 7, 2, 5, 1, 0, 9]

print("Desglose de audiencia por partido:")
print("-" * 75)

ocupacion = Counter(asignacion)
total = 0.0

for i, h in enumerate(asignacion):
    cat1, cat2 = partidos_ejemplo[i]
    dia, hora = DIAS_HORARIOS[h]

    base = audiencia_base(cat1, cat2)
    pond = PONDERACION[DIAS_HORARIOS[h]]
    coinc = ocupacion[h] - 1
    fc = factor_coincidencia(coinc)
    aud = base * pond * fc
    total += aud

    print(f"  P{i} ({cat1}v{cat2}) | {dia} {hora}h | base={base:.2f} × pond={pond:.2f} × coinc={fc:.2f} = {aud:.4f}M")

print("-" * 75)
print(f"  AUDIENCIA TOTAL: {total:.4f} millones")
print(f"  (verificación con la función: {audiencia_total(asignacion, partidos_ejemplo):.4f}M)")

Desglose de audiencia por partido:
---------------------------------------------------------------------------
  P0 (AvA) | Sabado 20h | base=2.00 × pond=1.00 × coinc=1.00 = 2.0000M
  P1 (AvB) | Domingo 20h | base=1.30 × pond=1.00 × coinc=1.00 = 1.3000M
  P2 (AvB) | Sabado 18h | base=1.30 × pond=0.80 × coinc=1.00 = 1.0400M
  P3 (BvB) | Domingo 16h | base=0.90 × pond=0.75 × coinc=1.00 = 0.6750M
  P4 (BvB) | Domingo 18h | base=0.90 × pond=0.85 × coinc=1.00 = 0.7650M
  P5 (BvC) | Sabado 16h | base=0.75 × pond=0.70 × coinc=1.00 = 0.5250M
  P6 (BvC) | Domingo 12h | base=0.75 × pond=0.45 × coinc=1.00 = 0.3375M
  P7 (CvC) | Sabado 12h | base=0.47 × pond=0.55 × coinc=1.00 = 0.2585M
  P8 (CvC) | Viernes 20h | base=0.47 × pond=0.40 × coinc=1.00 = 0.1880M
  P9 (CvC) | Lunes 20h | base=0.47 × pond=0.40 × coinc=1.00 = 0.1880M
---------------------------------------------------------------------------
  AUDIENCIA TOTAL: 7.2770 millones
  (verificación con la función: 7.2770M)


In [22]:
# Comparación rápida
# para ver que efectivamente queremos MAXIMIZAR

buena = [4, 8, 3, 6, 7, 2, 5, 1, 0, 9]   # repartida
mala  = [0, 0, 0, 0, 0, 9, 9, 9, 9, 9]     # todo apilado en viernes y lunes

print(f"Asignación repartida:  {audiencia_total(buena, partidos_ejemplo):.4f}M")
print(f"Asignación apilada:    {audiencia_total(mala, partidos_ejemplo):.4f}M")
print(f"Queremos maximizar: buscar la asignación con mayor audiencia.")

Asignación repartida:  7.2770M
Asignación apilada:    1.1172M
Queremos maximizar: buscar la asignación con mayor audiencia.


<br> **4. Diseña un algoritmo para resolver el problema por fuerza bruta**

**Respuesta:**

La idea de la fuerza bruta es: probar **todas** las asignaciones posibles, quedarnos solo con las que cumplen las restricciones (viernes y lunes obligatorios), y de esas elegir la que tenga mayor audiencia.

Usamos `itertools.product(range(10), repeat=n)` para generar todas las combinaciones de horarios para los $n$ partidos. Para cada una comprobamos si es válida con `es_valida()`, y si lo es calculamos su audiencia con `audiencia_total()`.

El problema es que con 10 partidos tendríamos $10^{10} = 10.000.000.000$ combinaciones, que es inviable. Por eso, para demostrar que el algoritmo funciona, lo ejecutamos con un número reducido de partidos (5 o 6) y dejamos constancia de que no escala al tamaño real.

In [23]:
def fuerza_bruta(partidos):
    """Prueba todas las asignaciones posibles y devuelve la mejor válida."""
    n = len(partidos)
    mejor_asignacion = None
    mejor_audiencia = -1
    evaluadas = 0

    for asig in itertools.product(range(10), repeat=n):
        asig = list(asig)
        if not es_valida(asig):
            continue
        evaluadas += 1
        aud = audiencia_total(asig, partidos)
        if aud > mejor_audiencia:
            mejor_audiencia = aud
            mejor_asignacion = asig[:]

    return mejor_asignacion, mejor_audiencia, evaluadas


# Probamos con pocos partidos para que sea viable
n_demo = 5
partidos_demo = partidos_ejemplo[:n_demo]

print(f"Fuerza bruta con {n_demo} partidos ({10**n_demo:,} combinaciones totales)")
print(f"Partidos: {partidos_demo}")
print()

inicio = time.time()
mejor_asig, mejor_aud, n_eval = fuerza_bruta(partidos_demo)
tiempo = time.time() - inicio

print(f"Soluciones válidas evaluadas: {n_eval:,}")
print(f"Tiempo: {tiempo:.2f} segundos")
print(f"\nMejor asignación: {mejor_asig}")
print(f"Audiencia máxima: {mejor_aud:.4f} millones")

# Mostramos la asignación
print("\nDetalles:")
for i, h in enumerate(mejor_asig):
    dia, hora = DIAS_HORARIOS[h]
    cat1, cat2 = partidos_demo[i]
    print(f"  Partido {i} ({cat1} vs {cat2}) -> {dia} {hora}h")

Fuerza bruta con 5 partidos (100,000 combinaciones totales)
Partidos: [('A', 'A'), ('A', 'B'), ('A', 'B'), ('B', 'B'), ('B', 'B')]

Soluciones válidas evaluadas: 14,670
Tiempo: 0.14 segundos

Mejor asignación: [4, 7, 8, 0, 9]
Audiencia máxima: 5.1250 millones

Detalles:
  Partido 0 (A vs A) -> Sabado 20h
  Partido 1 (A vs B) -> Domingo 18h
  Partido 2 (A vs B) -> Domingo 20h
  Partido 3 (B vs B) -> Viernes 20h
  Partido 4 (B vs B) -> Lunes 20h


In [8]:
# Veamos cómo crece el tiempo al aumentar el número de partidos

print("Escalabilidad de la fuerza bruta:")
print("-" * 50)

for n in range(3, 7):
    partidos_test = partidos_ejemplo[:n]
    inicio = time.time()
    _, mejor_aud, n_eval = fuerza_bruta(partidos_test)
    t = time.time() - inicio
    print(f"  n={n}: {10**n:>10,} combinaciones, {n_eval:>8,} válidas, {t:.3f}s -> aud={mejor_aud:.4f}M")

print(f"\n  n=10: {10**10:>10,} combinaciones -> INVIABLE")
print("  (tardaría horas o días)")

Escalabilidad de la fuerza bruta:
--------------------------------------------------
  n=3:      1,000 combinaciones,       54 válidas, 0.001s -> aud=3.0400M
  n=4:     10,000 combinaciones,      974 válidas, 0.010s -> aud=4.1800M
  n=5:    100,000 combinaciones,   14,670 válidas, 0.232s -> aud=5.1250M
  n=6:  1,000,000 combinaciones,  199,262 válidas, 4.188s -> aud=5.7850M

  n=10: 10,000,000,000 combinaciones -> INVIABLE
  (tardaría horas o días)


<br>**5. Calcula la complejidad del algoritmo por fuerza bruta**

**Respuesta:**

El algoritmo de fuerza bruta tiene que:

1. Generar todas las asignaciones posibles: hay $k^n$ (con $k=10$ horarios y $n$ partidos).
2. Para cada asignación, comprobar si es válida: esto es $O(n)$ (recorrer la lista buscando el viernes y el lunes).
3. Para cada asignación válida, **calcular** la audiencia total: también $O(n)$ (sumar la audiencia de cada partido).

Como los pasos 2 y 3 son $O(n)$ y se repiten $k^n$ veces, la complejidad total es:

$$O(k^n \cdot n)$$

Para nuestro caso concreto ($k=10$, $n=10$), eso son $10^{10} \times 10 = 10^{11}$ operaciones. Es un crecimiento **exponencial** en $n$, lo que lo hace completamente inviable para el tamaño real del problema.

Como hemos visto en la celda anterior, ya con 6 partidos el tiempo empieza a notarse. Con 10 sería impracticable.

In [9]:
# Estimación del tiempo que tardaría la fuerza bruta con n=10

# Medimos el tiempo para n=5 y extrapolamos
partidos_5 = partidos_ejemplo[:5]
inicio = time.time()
fuerza_bruta(partidos_5)
t5 = time.time() - inicio

# Extrapolamos: el tiempo crece como 10^n, así que t(10) ≈ t(5) * 10^5
t10_estimado = t5 * (10**5)  # 10^10 / 10^5 = 10^5 veces más

print(f"Tiempo medido con n=5: {t5:.3f} segundos")
print(f"Tiempo estimado con n=10: {t10_estimado:,.0f} segundos")
print(f"                        = {t10_estimado/3600:,.1f} horas")
print(f"                        = {t10_estimado/86400:,.1f} días")
print(f"\nClaramente inviable. Necesitamos un algoritmo mejor.")

Tiempo medido con n=5: 0.342 segundos
Tiempo estimado con n=10: 34,228 segundos
                        = 9.5 horas
                        = 0.4 días

Claramente inviable. Necesitamos un algoritmo mejor.


<br>**6. (*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta.**

**Respuesta:**<br>

Para mejorar la complejidad del algoritmo por fuerza bruta, proponemos un algoritmo heurístico constructivo de tipo Greedy (voraz). Este algoritmo se divide en dos fases:

1. Primero, nos aseguramos de cumplir las restricciones obligatorias y asignamos los dos partidos con menor audiencia base a los horarios obligatorios del viernes y lunes, que son los más penalizados. Emparejar los coeficientes más bajos con los valores base más pequeños asegura que el impacto negativo en la función objetivo sea el mínimo posible para cumplir la restricción.

2. Después, ordenamos los partidos restantes de mayor a menor audiencia base. Para cada uno de ellos, evaluamos los 10 horarios disponibles y le asignamos definitivamente el que le proporcione la mayor audiencia parcial en ese momento, teniendo en cuenta la penalización por coincidencias con los partidos que ya hemos colocado.

<br>Mientras que la fuerza bruta evalúa todas las combinaciones posibles, este algoritmo toma decisiones iterativas y sin retroceso, por lo que solo evalúa $n \times k$ opciones,  que en este caso, como tenemos 10 partidos y 10 horarios serían 100 opciones en total.

Este algoritmo no nos garantiza encontrar la solución óptima global ya que puede quedarse en un óptimo local, pero la idea de asignar primero los partidos más importantes a los mejores horarios es intuitiva y suele dar buenos resultados en la práctica.

In [10]:
def greedy(partidos):
    n = len(partidos)
    asignacion = [None] * n # inicializamos la lista de la solución
    ocupacion = Counter()

    # Ordenamos los partidos por audiencia base de mayor a menor
    indices_ordenados = sorted(range(n), key=lambda i: audiencia_base(*partidos[i]), reverse=True)

    # Fase 1: asignar los dos peores partidos a viernes y lunes
    peor_1 = indices_ordenados[-1]  # menor audiencia base
    peor_2 = indices_ordenados[-2]  # segundo menor

    asignacion[peor_1] = IDX_VIERNES
    ocupacion[IDX_VIERNES] += 1
    asignacion[peor_2] = IDX_LUNES
    ocupacion[IDX_LUNES] += 1

    # Paso 2: asignar el resto de forma voraz
    restantes = [i for i in indices_ordenados if asignacion[i] is None]

    for i in restantes:
        mejor_horario = -1
        mejor_aud = -1

        # Probamos cada horario y nos quedamos con el mejor
        for h in range(len(DIAS_HORARIOS)):
            base = audiencia_base(*partidos[i])
            pond = PONDERACION[DIAS_HORARIOS[h]]
            coinc = ocupacion[h]  # partidos ya en ese horario
            fc = factor_coincidencia(coinc)
            aud = base * pond * fc

            if aud > mejor_aud:
                mejor_aud = aud
                mejor_horario = h

        asignacion[i] = mejor_horario
        ocupacion[mejor_horario] += 1

    return asignacion

In [11]:
# Probamos con los 10 partidos
inicio = time.time()
asig_greedy = greedy(partidos_ejemplo)
tiempo_greedy = time.time() - inicio
aud_greedy = audiencia_total(asig_greedy, partidos_ejemplo)

print("Resultado del algoritmo Greedy (10 partidos):")
print("-" * 55)
for i, h in enumerate(asig_greedy):
    dia, hora = DIAS_HORARIOS[h]
    cat1, cat2 = partidos_ejemplo[i]
    print(f"  Partido {i} ({cat1} vs {cat2}) -> {dia} {hora}h")

print(f"\nAudiencia total: {aud_greedy:.4f} millones")
print(f"Tiempo: {tiempo_greedy:.6f} segundos")
print(f"¿Válida? {es_valida(asig_greedy)}")

Resultado del algoritmo Greedy (10 partidos):
-------------------------------------------------------
  Partido 0 (A vs A) -> Sabado 20h
  Partido 1 (A vs B) -> Domingo 20h
  Partido 2 (A vs B) -> Domingo 18h
  Partido 3 (B vs B) -> Sabado 18h
  Partido 4 (B vs B) -> Sabado 20h
  Partido 5 (B vs C) -> Domingo 16h
  Partido 6 (B vs C) -> Domingo 20h
  Partido 7 (C vs C) -> Sabado 16h
  Partido 8 (C vs C) -> Lunes 20h
  Partido 9 (C vs C) -> Viernes 20h

Audiencia total: 6.8050 millones
Tiempo: 0.000401 segundos
¿Válida? True


 A continuación, vamos a realizar una comparación contra la fuerza bruta. Dado que la fuerza bruta es inviable para el tamaño real del problema, ejecutaremos la prueba para un subconjunto de 5 partidos.

In [12]:
# Comparación
partidos_5 = partidos_ejemplo[:5]

# Fuerza bruta
inicio = time.time()
asig_fb, aud_fb, _ = fuerza_bruta(partidos_5)
t_fb = time.time() - inicio

# Nuestro algoritmo Greedy
inicio = time.time()
asig_gr = greedy(partidos_5)
t_gr = time.time() - inicio
aud_gr = audiencia_total(asig_gr, partidos_5)

print(f"Comparación con {len(partidos_5)} partidos:")
print(f"  Fuerza bruta: {aud_fb:.4f}M en {t_fb:.4f}s (óptimo garantizado)")
print(f"  Greedy:       {aud_gr:.4f}M en {t_gr:.6f}s")
print(f"\n  Diferencia de audiencia: {aud_fb - aud_gr:.4f}M")
print(f"  El greedy es {t_fb/max(t_gr, 1e-9):.0f}x más rápido")

Comparación con 5 partidos:
  Fuerza bruta: 5.1250M en 0.3154s (óptimo garantizado)
  Greedy:       5.1250M en 0.000208s

  Diferencia de audiencia: 0.0000M
  El greedy es 1515x más rápido


<br>**7. (*)Calcula la complejidad del algoritmo.**

**Respuesta:**

El algoritmo greedy tiene que:

1. Ordenar la lista de los $n$ partidos según su audiencia base. El algoritmo de ordenación nativo de Python (`sorted()`) utiliza un método llamado Timsort, lo que nos da una complejidad de $O(n \log n)$.
2. Asignar viernes y lunes es una operación de $O(1)$.
3. Finalmente, para cada uno de los $n-2$ partidos restantes, el algoritmo itera a través de los $k$ horarios disponibles evaluando la función objetivo, lo cual es una operación matemática simple $O(1)$. Por tanto, este bucle tiene una complejidad de $O(n \cdot k)$ y dado que  $k$ es una constante fija, $O(n \cdot k)$ se comporta como $O(n)$.

En total, la complejidad del algoritmo es la suma de todas las partes, esto es, $O(n \log n + n)$, que se simplifica a $O(n \log n)$, ya que se descartan los términos de menor orden para quedarnos únicamente con el término dominante a medida que $n$ tiende a infinito.

Para nuestro caso con $n=10$ y $k=10$, el coste es de unas 100 operaciones (más la pequeña ordenación inicial), lo cual es casi despreciable frente a las $10^{11}$ operaciones de la fuerza bruta. La diferencia es muy significativa: pasamos de un crecimiento exponencial ($O(k^n \cdot n)$) a un tiempo cuasilineal $O(n \log n)$, resolviendo el problema de forma instantánea.

| | Fuerza bruta | Greedy |
|:---|:---:|:---:|
| **Complejidad** | $O(k^n \cdot n)$ | $O(n\log{n})$ |
| **Operaciones ($n=10, k=10$)** | $\sim 10^{11}$ | $\sim 100$ |
| **Óptimo garantizado** | Sí | No |

Para ilustrar esta eficiencia computacional, la siguiente prueba evalúa el algoritmo Greedy con volúmenes de datos crecientes. Esto demuestra en la práctica cómo la solución escala sin problemas hasta miles de partidos en fracciones de segundo.

In [13]:
# Verificación
print("Tiempo del greedy al aumentar n:")
print("-" * 40)

for n in [5, 10, 50, 100, 500, 1000]:
    # Generamos partidos aleatorios
    cats = ['A', 'B', 'C']
    partidos_test = [(random.choice(cats), random.choice(cats)) for _ in range(n)]

    inicio = time.time()
    asig = greedy(partidos_test)
    t = time.time() - inicio

    print(f"  n={n:>5}: {t:.6f}s")

Tiempo del greedy al aumentar n:
----------------------------------------
  n=    5: 0.000073s
  n=   10: 0.000113s
  n=   50: 0.000800s
  n=  100: 0.001441s
  n=  500: 0.005462s
  n= 1000: 0.011069s


<br>**8. Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios.**

**Respuesta:**

Para generar jornadas aleatorias, respetamos la distribución de categorías del enunciado. La Liga tiene 20 equipos, de los cuales 3 son de categoría A, 11 de categoría B y 6 de categoría C.

En cada jornada, cada equipo juega exactamente un partido, así que tenemos 10 partidos. Vamos a crear la lista de 20 equipos con sus categorías, después la barajamos aleatoriamente, emparejamos los equipos de dos en dos y extraemos las categorías de cada pareja.

In [14]:
def generar_jornada_aleatoria(seed=None):
    """Genera una jornada aleatoria respetando las categorías de La Liga."""
    if seed is not None:
        random.seed(seed)

    # 20 equipos: 3A, 11B, 6C
    equipos = ['A'] * 3 + ['B'] * 11 + ['C'] * 6
    random.shuffle(equipos)

    # Emparejamos de dos en dos
    partidos = []
    for j in range(0, 20, 2):
        cat1, cat2 = equipos[j], equipos[j+1]
        partidos.append(tuple(sorted([cat1, cat2])))  # ordenamos para consistencia

    return partidos


# Generamos varias jornadas de ejemplo
print("Ejemplos de jornadas aleatorias:")
print("=" * 55)

for j in range(5):
    partidos = generar_jornada_aleatoria(seed=j)
    print(f"\nJornada {j+1}: {partidos}")

Ejemplos de jornadas aleatorias:

Jornada 1: [('B', 'C'), ('C', 'C'), ('A', 'C'), ('A', 'B'), ('B', 'B'), ('B', 'B'), ('B', 'C'), ('B', 'C'), ('A', 'B'), ('B', 'B')]

Jornada 2: [('B', 'B'), ('C', 'C'), ('A', 'B'), ('A', 'C'), ('B', 'C'), ('B', 'B'), ('B', 'C'), ('B', 'B'), ('A', 'B'), ('B', 'C')]

Jornada 3: [('B', 'B'), ('B', 'C'), ('C', 'C'), ('A', 'B'), ('B', 'B'), ('B', 'C'), ('B', 'B'), ('B', 'C'), ('B', 'C'), ('A', 'A')]

Jornada 4: [('B', 'B'), ('B', 'B'), ('C', 'C'), ('A', 'B'), ('A', 'A'), ('B', 'B'), ('B', 'C'), ('B', 'C'), ('B', 'C'), ('B', 'C')]

Jornada 5: [('C', 'C'), ('B', 'C'), ('B', 'C'), ('B', 'C'), ('B', 'B'), ('A', 'B'), ('A', 'B'), ('A', 'C'), ('B', 'B'), ('B', 'B')]


<br>**9. Aplica el algoritmo al juego de datos generado.**

**Respuesta:**

Vamos a ejecutar el algoritmo Greedy sobre 10 jornadas completas generadas aleatoriamente. El objetivo es verificar que, ante cualquier combinación posible de enfrentamientos, el algoritmo es capaz de devolver una asignación válida (cumpliendo las restricciones de viernes y lunes) en un tiempo de ejecución constante y mínimo.

In [15]:
print("Greedy aplicado a 10 jornadas aleatorias (10 partidos cada una):")
print("=" * 65)

for j in range(10):
    partidos = generar_jornada_aleatoria(seed=j)

    inicio = time.time()
    asig = greedy(partidos)
    t = time.time() - inicio
    aud = audiencia_total(asig, partidos)

    print(f"  Jornada {j+1:>2}: audiencia = {aud:.4f}M | tiempo = {t:.6f}s | válida = {es_valida(asig)}")

Greedy aplicado a 10 jornadas aleatorias (10 partidos cada una):
  Jornada  1: audiencia = 6.4455M | tiempo = 0.000135s | válida = True
  Jornada  2: audiencia = 6.4455M | tiempo = 0.000107s | válida = True
  Jornada  3: audiencia = 6.7730M | tiempo = 0.000101s | válida = True
  Jornada  4: audiencia = 6.7730M | tiempo = 0.000099s | válida = True
  Jornada  5: audiencia = 6.4455M | tiempo = 0.000107s | válida = True
  Jornada  6: audiencia = 6.3330M | tiempo = 0.000098s | válida = True
  Jornada  7: audiencia = 6.6600M | tiempo = 0.000098s | válida = True
  Jornada  8: audiencia = 6.4455M | tiempo = 0.000113s | válida = True
  Jornada  9: audiencia = 6.7735M | tiempo = 0.000098s | válida = True
  Jornada 10: audiencia = 6.4460M | tiempo = 0.000097s | válida = True


Finalmente, seleccionamos una jornada concreta para observar cómo el algoritmo ha distribuido la carga de partidos.

In [16]:
# Detalle de una jornada concreta para ver la asignación completa
partidos_detalle = generar_jornada_aleatoria(seed=42)
asig_detalle = greedy(partidos_detalle)

print("Detalle de la jornada (seed=42):")
print("-" * 55)
for i, h in enumerate(asig_detalle):
    dia, hora = DIAS_HORARIOS[h]
    cat1, cat2 = partidos_detalle[i]
    print(f"  Partido {i} ({cat1} vs {cat2}) -> {dia} {hora}h")

print(f"\nAudiencia total: {audiencia_total(asig_detalle, partidos_detalle):.4f} millones")
print(f"¿Válida? {es_valida(asig_detalle)}")

Detalle de la jornada (seed=42):
-------------------------------------------------------
  Partido 0 (B vs C) -> Domingo 20h
  Partido 1 (B vs C) -> Sabado 16h
  Partido 2 (B vs B) -> Sabado 18h
  Partido 3 (C vs C) -> Viernes 20h
  Partido 4 (B vs B) -> Sabado 20h
  Partido 5 (B vs C) -> Lunes 20h
  Partido 6 (A vs B) -> Sabado 20h
  Partido 7 (A vs C) -> Domingo 18h
  Partido 8 (B vs B) -> Domingo 16h
  Partido 9 (A vs B) -> Domingo 20h

Audiencia total: 6.4455 millones
¿Válida? True


Podemos ver que los horarios obligatorios y peor ponderados (viernes y lunes) absorben exactamente un partido de baja categoría, mientras que el resto se agrupa de forma inteligente en los horarios de fin de semana intentando minimizar la penalización por coincidencia.

<br>**10. Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo**

**Respuesta:**
- Material de la asignatura 03MIAR – Algoritmos de Optimización (VIU): transparencias y vídeos sobre algoritmos greedy, fuerza bruta y complejidad computacional.
- Documentación de Python: módulos itertools, collections.Counter, random y time. https://docs.python.org/3/library/
- Cormen, T. H., Leiserson, C. E., Rivest, R. L., & Stein, C. (2009). *Introduction to Algorithms* (3rd ed.). MIT Press. Capítulos sobre algoritmos greedy y análisis de complejidad.
- Asistencia de Claude (Anthropic) para la revisión y estructuración de las respuestas.

<br>**11. Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño.**

**Respuesta:**

Trabajando en este seminario han surgido varias formas de seguir explorando el problema:

Lo más directo sería probar otros algoritmos. El greedy funciona bien y es muy rápido, pero se sabe que no garantiza la mejor solución posible. Investigando un poco, se han encontrado técnicas como Simulated Annealing o algoritmos genéticos, que podrían dar mejores resultados al explorar más combinaciones sin llegar al extremo de la fuerza bruta. Otra opción sería usar la solución del greedy como punto de partida y aplicar búsqueda local: ir cambiando partidos de horario uno a uno mientras la audiencia mejore.

También sería interesante acercar el problema a la realidad. Aquí solo se tienen dos restricciones (viernes y lunes obligatorios), pero en la vida real hay muchas más: los derbis no pueden coincidir, equipos de la misma ciudad no juegan a la vez, hay partidos que van en abierto y tienen horario fijo, descansos entre jornadas, etc. Meter todo eso complicaría bastante el modelo pero lo haría mucho más útil.

Otro punto es el tema del tamaño. Aquí se trabaja con una sola jornada (10 partidos, 10 horarios), pero si se quisiera planificar varias jornadas a la vez el espacio de soluciones crecería enormemente. Ahí ya se entraría en terreno de programación lineal entera con herramientas tipo CPLEX o Gurobi.

Por último, los datos de audiencia utilizados son estimaciones fijas. Con datos reales y un modelo predictivo que tenga en cuenta cosas como la clasificación actual, si es un derbi, o si coincide con otros eventos, los resultados serían mucho más realistas.